# Versioned polygenic-score calculation

M7 uses explicitly selected local GRCh38 additive models and exact canonical alleles. Missing markers stay missing; partial sums are not published complete scores. Raw scores are not probabilities, diagnoses, absolute risks, treatment advice, or portable percentiles. Personal contribution artifacts remain private. Colab is a Google-managed VM; use the local workflow if cloud processing is unacceptable.


In [ ]:
import os
import sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
from genome_evidence.notebook_support import resolve_settings
SETTINGS = resolve_settings()
PROFILE = SETTINGS.profile
REPOSITORY_URL = SETTINGS.repository_url
REPOSITORY_REF = SETTINGS.repository_ref
WORKSPACE_ROOT = SETTINGS.workspace_root
SUBJECT_ID = SETTINGS.subject_id
print({"profile": PROFILE, "requested_ref": REPOSITORY_REF})


In [ ]:
if PROFILE == "personal_drive":
    from genome_evidence.workspace import validate_workspace

    validate_workspace(WORKSPACE_ROOT)
else:
    assert PROFILE == "synthetic_ci"

In [ ]:
PGS_IDS = []
SCORE_BUNDLE = None
GENOTYPE_SOURCE_POLICY = "observed_only"
IMPUTATION_RUN = None
REFERENCE_DISTRIBUTION = None
if PROFILE == "personal_drive" and not PGS_IDS:
    raise ValueError(
        "PGS_IDS is required: select explicit checked local model IDs, then rerun notebook 06"
    )

In [ ]:
from genome_evidence.polygenic_scoring.models import ScoreConfig
from genome_evidence.polygenic_scoring.reference import compare_reference

if PROFILE == "synthetic_ci":
    config = ScoreConfig(pgs_ids=("PGS999999",))
    identity = {
        "pgs_id": "PGS999999",
        "model_version": "synthetic-v1",
        "assembly": "GRCh38",
        "matching_pipeline": "exact-v1",
        "missingness_policy": "all-supported",
    }
    compatible = compare_reference(1.5, {**identity, "scores": [0.0, 1.0, 2.0, 3.0]}, identity)
    incompatible = compare_reference(
        1.5, {**identity, "assembly": "GRCh37", "scores": [0.0, 1.0]}, identity
    )
    assert compatible["status"] == "evaluable"
    assert incompatible["status"] == "not_evaluable"
    print(
        {
            "model": "PGS999999",
            "fixture": "fabricated",
            "source": "observed",
            "coverage_status": "partial_not_evaluable",
            "reference_statuses": [compatible["status"], incompatible["status"]],
        }
    )

## Next step
Personal mode requires a compatible completed M2 run and an explicitly selected, validated local score bundle. No network acquisition occurs during analysis.
